In [ ]:
from datetime import datetime
import pandas as pd
import logging
from pathlib import Path
from sqlalchemy import create_engine, inspect, URL, select, func, text
import re


TODAY = datetime.now().strftime("%Y-%m-%d")
PROJECT_ROOT = Path.cwd().parents[1]


log_dir = PROJECT_ROOT / "logs" / "local_scraper"
log_dir.mkdir(exist_ok=True)
log_file = log_dir / f"scraper_{TODAY}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.FileHandler(log_file),
              logging.StreamHandler()],
    force=True
)

logger = logging.getLogger(__name__)

In [17]:
def get_columns_from_table(inspector, table: str, schema: str ='public') -> set:
    return {col["name"] for col in inspector.get_columns(table, schema=schema)}

def send_raw_parquet_to_landing(engine, listing_path: str) -> int | None:
    df = pd.read_parquet(listing_path)

    res = df.to_sql(
        name="landing",
        con=engine,
        schema="staging",
        if_exists="append",
        index=False
                )

    return res

def get_unloaded_scrape_dates(max_date: str) -> str:

    project_root = Path.cwd().parents[1]
    bucket_dir = project_root / "raw_bucket" 

    # only folder names that match regex and bigger than max date
    unloaded_dates = []

    for item in bucket_dir.iterdir():
        if item.is_dir() and re.fullmatch(r"^\d{4}-\d{2}-\d{2}$", item.name) and item.name > max_date:
            unloaded_dates.append(item.name)

    unloaded_paths = [bucket_dir / date for date in unloaded_dates]

    unloaded_files = [[file for file in folder.iterdir() if file.is_file() and file.suffix == ".parquet"] for folder in unloaded_paths]

    latest_scrapes_from_folders = [lst[0] for lst in unloaded_files]

    return latest_scrapes_from_folders

In [18]:
def main():
    url_object = URL.create(
        "postgresql+psycopg",
        username="postgres",
        password="admin",
        host="localhost",
        database="main"
    )
    engine = create_engine(url_object)
    inspector = inspect(engine)
    
    with engine.connect() as conn:
        landing_max_date = conn.scalar(text("SELECT MAX(scraped_at) from staging.landing"))
        logger.info(f"Latest date from 'landing' - {landing_max_date}")

    # if landing_max_date == TODAY:
    if landing_max_date < TODAY:
        to_load = get_unloaded_scrape_dates(landing_max_date)
        # to_load = get_unloaded_scrape_dates('2026-07-14')
        
        if len(to_load) > 0:
            logger.info(f"Loading {len(to_load)} file/s")

            try:
                for file in to_load:
                    if send_raw_parquet_to_landing(engine, file) == -1:
                        logger.info(f'Succesfully loaded {file}')
            except Exception as e:
                logger.error(f'ERROR - {e} on loading {file}')
        else:
            logger.info(f"All available files are loaded")
    else:
        logger.info(f"Database is up to date with all raw files")

In [19]:
main()

2026-07-28 00:23:34,829 [INFO] Latest date from 'landing' - 2026-07-27
2026-07-28 00:23:34,831 [INFO] All available files are loaded
